In [38]:
from FormUtils import pyForm, capture_physics_expr

In [39]:
%%pyForm Renormalization

* Process: Renormalization

#-
* Above suppresses extra output
Off Statistics;
Off FinalStats;
#include SquareAmplitude.h

Symbols e, Mmuon, Melec, x;
Vectors k, kf1, kf2;
Symbols k2 , kminusq2, kdotp1, kminusqdotp2, kdotkminusq;


Local MLO = (e^2) * (UB(i1, p3, Melec ) * g(i1, i2, mu1) * U(i2, p1, Melec))
            * phprop(mu1, mu2, q)
            * (UB(i3, p4, Mmuon) * g(i3, i4, mu2) * U(i4, p2, Mmuon));

Local MNLO = -1* (e^4) 
             * (UB(i1, p3, Melec) * g(i1, i2, mu1) * U(i2, p1, Melec)) 
             * phprop(mu1, mu3, q) 
             * g(i11, i12, mu3) * fprop(i12, i13, kf1, Melec) 
             * g(i13, i14, mu4) * fprop(i14, i11, kf2, Melec)
             * phprop(mu4, mu2, q)
             * (UB(i7, p4, Mmuon) * g(i7, i8, mu2) * U(i8, p2, Mmuon));

Local MTotal = MLO+MNLO;
#call squareamplitude(MLO, MsqLO)
#call squareamplitude(MNLO, MsqNLO)
#call squareamplitude(MTotal, MsqTotal)
.sort
Multiply 1/4;
.sort
Drop drop MsqNLO, MsqTotal, MLO, MNLO, MTotal ;
Local MInt = MsqTotal - MsqLO - MsqNLO;
.sort


* --- KINEMATIC DEFINITIONS: VACUUM POLARIZATION ---
* q  = p1 - p3           : Momentum transfer between electron and muon lines
* t  = q.q               : Mandelstam variable t (photon momentum squared)
* k  = loop momentum     : Integration variable for the fermion loop
* kf1 = k                : Momentum of the first fermion in the bubble
* kf2 = k - q            : Momentum of the second fermion in the bubble (cons. of momentum)

* Replace the propagator function with algebraic denominators
id prop(q.q) = 1/t;
id prop(-Melec^2 + kf1.kf1) = 1/(-Melec^2 +k2);
id prop(-Melec^2 + kf2.kf2) = 1/(-Melec^2 +kminusq2);

* Replace dot products involving loop momentum k
id kf1.p1? = kdotp1;
id kf2.p2? = kminusqdotp2;
id kf1.kf2 = kdotkminusq;
.sort

* --- MASSLESS APPROXIMATION ---
* keeps the Melec in the fermion propagator
id Melec = 0;
id Mmuon = 0;
#call Mandelstam2To2(p1,p2,p3,p4,0,0,0,0)
id u = -s -t;

* Save to file 
Format C;
#write <RenormalizationLO.txt> "%e;", MsqLO;
#write <Renormalization.txt> "%e;", MInt;
.sort
* Print 
Format;
Print+s MInt;
Print+s MsqLO;

.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Thu Apr 16 22:34:19 2026
    
    * Process: Renormalization
    
    #-

   MsqLO =
       + 2*e^4
       + 4*s*t^(-1)*e^4
       + 4*s^2*t^(-2)*e^4
      ;

   MInt =
       + 192/(k2 - Melec^2)/(kminusq2 - Melec^2)*t^(-2)*e^6*kdotp1*
      kminusqdotp2
       + 128/(k2 - Melec^2)/(kminusq2 - Melec^2)*s*t^(-3)*e^6*kdotp1*
      kminusqdotp2
       - 32/(k2 - Melec^2)/(kminusq2 - Melec^2)*s*t^(-2)*e^6*kdotp1
       - 32/(k2 - Melec^2)/(kminusq2 - Melec^2)*s^2*t^(-3)*e^6*kdotp1
      ;




In [44]:
import numpy as np
import sympy as sp

form_expr = capture_physics_expr("scripts/RenormalizationLO.txt")
s, t = sp.symbols("s t")
MLO = sp.expand(form_expr)
print(f"MLO =  {MLO}\n")

form_expr = capture_physics_expr("scripts/Renormalization.txt")
Melec, s, t, k2, kminusq2, kdotp1, kminusqdotp2 = sp.symbols(
    "Melec s t  k2 kminusq2  kdotp1 kminusqdotp2"
)
D1, D2 = sp.symbols('D1 D2')
MInt_symbolic = form_expr.subs({
    (k2 - Melec**2): D1,
    (kminusq2 - Melec**2): D2
})

MInt_collected = sp.collect(MInt_symbolic, [s, t])
print(f"MInt_collected =  {MInt_collected}\n")

MInt_final = sp.factor(MInt_collected)
print(f"MInt_final = {MInt_final}\n")

# x: Feynman parameter, eps: 4-d,
# mu2: renormalization scale, Delta: loop denominator
x, eps, mu2 = sp.symbols("x eps mu2", real=True)

# Feynman parametrization: Delta = m^2 - x(1-x)q^2
# t-channel process: q^2 = t.
Delta = Melec**2 - x*(1-x)*t

# See 'Diagrammatica'
# Shift k -> l + xq to center the denominator.
# Terms linear in the new loop momentum 'l' vanish due to parity.
# Kinematic replacements for t-channel: q.p1 = t/2 and q.p2 = -t/2.
shift_rules = {
    kdotp1: x * (t/2),               # Contribution from (l + xq).p1
    kminusqdotp2: (x - 1) * (-t/2)   # Contribution from (l + (x-1)q)
}
# Apply the shift to expanded MInt
MInt_shifted = MInt_expanded.subs(shift_rules)

# Result of Integral[ d^d l / (2*pi)^d * 1/(l^2 - Delta)^2 ]
# We focus on the (2/eps - log(Delta)) part which contains the physics.
master_integral = (2/eps - sp.log(Delta/mu2))
MInt_numerator = MInt_shifted.subs({k2: 0, kminusq2: 0})

MInt_momentum_integrated = MInt_numerator * master_integral
print(f"MInt_momentum_integrated = {MInt_momentum_integrated}\n")



MLO =  2*e**4*s**2/t**2 + 2*e**4*u**2/t**2



AttributeError: 'str' object has no attribute 'subs'